# Boosting Models

This notebook evaluates gradient boosting models on the Titanic dataset.

Models:
- CatBoost
- LightGBM
- XGBoost

Goals:
- Establish baseline performance for each boosting algorithm.
- Explore the main hyperparameters.
- Compare different approaches to categorical features.
- Tune the models using cross-validation.
- Compare the results with classical ML baselines.

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
#%cd ..

c:\Users\coffe\Desktop\titanic-ml\titanic-ml


In [3]:
import pandas as pd

from sklearn.pipeline import Pipeline

from src.models import get_model
from src.preprocessing import build_preprocessor
from src.validation import evaluate_model

SEED = 42

In [4]:
train = pd.read_csv("data/train.csv")

X = train.drop(columns="Survived")
y = train["Survived"]

### CatBoost preprocessing strategy

CatBoost natively supports categorical features.
Categorical features will therefore not be one-hot encoded.

The preprocessing pipeline will only:
- impute missing Age values;
- impute missing Embarked values;
- create HasCabin;
- select the required features.

Categorical features will be passed directly to CatBoost.

In [5]:
catboost_processor = build_preprocessor('catboost')

X_test = catboost_processor.fit_transform(X, y)

X_test.head()

,Age,Fare,SibSp,Parch,Sex,Embarked,Pclass,HasCabin
0,22.0,7.2500,1,0,male,S,3,0
1,38.0,71.2833,1,0,female,C,1,1
2,26.0,7.9250,0,0,female,S,3,0
3,35.0,53.1000,1,0,female,S,1,1
4,35.0,8.0500,0,0,male,S,3,0


# Catboost approach

In [6]:
import pandas as pd

from src.catboost_validation import evaluate_catboost

SEED = 42

In [7]:
train = pd.read_csv("data/train.csv")

X = train.drop(columns="Survived")
y = train["Survived"]

In [ ]:
CAT_FEATURES = [
    "Sex",
    "Embarked",
    "Pclass",
]

result = evaluate_catboost(
    X=X,
    y=y,
    cat_features=CAT_FEATURES,
    seed=SEED,
)

result

In [ ]:
iterations_values = [10, 25, 50, 100, 200, 500, 1000]

for iterations in iterations_values:
    result = evaluate_catboost(
        X=X,
        y=y,
        cat_features=CAT_FEATURES,
        model_params={
            "iterations": iterations,
        },
        seed=SEED,
    )

    print(
        f"iterations ={iterations:4d} | "
        f"CV={result['mean']:.4f} ± {result['std']:.4f}"
    )

Fold 1: 0.8436
Fold 2: 0.8034
Fold 3: 0.8034
Fold 4: 0.8202
Fold 5: 0.8427
iterations=  10 | CV=0.8226 ± 0.0178
Fold 1: 0.8436
Fold 2: 0.8202
Fold 3: 0.7865
Fold 4: 0.8427
Fold 5: 0.8315
iterations=  25 | CV=0.8249 ± 0.0210
Fold 1: 0.8492
Fold 2: 0.8090
Fold 3: 0.7921
Fold 4: 0.8427
Fold 5: 0.8652
iterations=  50 | CV=0.8316 ± 0.0269
Fold 1: 0.8436
Fold 2: 0.8202
Fold 3: 0.7978
Fold 4: 0.8427
Fold 5: 0.8539
iterations= 100 | CV=0.8316 ± 0.0202
Fold 1: 0.8603
Fold 2: 0.8315
Fold 3: 0.7978
Fold 4: 0.8427
Fold 5: 0.8146
iterations= 200 | CV=0.8294 ± 0.0217
Fold 1: 0.8436
Fold 2: 0.8371
Fold 3: 0.7921
Fold 4: 0.8371
Fold 5: 0.8483
iterations= 500 | CV=0.8316 ± 0.0202
Fold 1: 0.8547
Fold 2: 0.8427
Fold 3: 0.7978
Fold 4: 0.8371
Fold 5: 0.8258
iterations=1000 | CV=0.8316 ± 0.0193


In [16]:
learning_rates = [0.01, 0.03, 0.05, 0.1, 0.2, 0.3]

for learning_rate in learning_rates:
    result = evaluate_catboost(
        X=X,
        y=y,
        cat_features=CAT_FEATURES,
        model_params={
            "iterations": 500,
            "learning_rate": learning_rate,
        },
        seed=SEED,
    )

    print(
        f"learning_rate = {learning_rate:.2f} | "
        f"CV={result['mean']:.4f} ± {result['std']:.4f}"
    )

Fold 1: 0.8436
Fold 2: 0.8258
Fold 3: 0.8146
Fold 4: 0.8371
Fold 5: 0.8371
learning_rate = 0.01 | CV=0.8316 ± 0.0103
Fold 1: 0.8715
Fold 2: 0.8427
Fold 3: 0.7921
Fold 4: 0.8483
Fold 5: 0.8258
learning_rate = 0.03 | CV=0.8361 ± 0.0264
Fold 1: 0.8603
Fold 2: 0.8427
Fold 3: 0.8034
Fold 4: 0.8146
Fold 5: 0.8539
learning_rate = 0.05 | CV=0.8350 ± 0.0223
Fold 1: 0.8603
Fold 2: 0.8427
Fold 3: 0.7978
Fold 4: 0.8202
Fold 5: 0.8315
learning_rate = 0.10 | CV=0.8305 ± 0.0211
Fold 1: 0.8603
Fold 2: 0.8483
Fold 3: 0.8090
Fold 4: 0.8034
Fold 5: 0.8539
learning_rate = 0.20 | CV=0.8350 ± 0.0239
Fold 1: 0.8659
Fold 2: 0.8427
Fold 3: 0.8202
Fold 4: 0.8034
Fold 5: 0.8539
learning_rate = 0.30 | CV=0.8372 ± 0.0227


### Learning rate tuning

The learning rate was evaluated using cross-validation while keeping
the remaining CatBoost parameters fixed.

The CV scores were relatively stable across the tested range.
Although `learning_rate=0.30` achieved the highest mean CV accuracy,
the difference compared with other values was small relative to the
cross-validation variability.

Therefore, the experiment does not provide strong evidence that a
single learning rate is clearly superior.

In [9]:
iterations_grid = [50, 100, 200, 500]
learning_rates = [0.01, 0.03, 0.05, 0.1, 0.2, 0.3]

for iterations in iterations_grid:
    for learning_rate in learning_rates:

        result = evaluate_catboost(
            X=X,
            y=y,
            cat_features=CAT_FEATURES,
            model_params={
                "iterations": iterations,
                "learning_rate": learning_rate,
            },
            seed=SEED,
        )

        print(
            f"iterations={iterations:3d} | "
            f"lr={learning_rate:.2f} | "
            f"CV={result['mean']:.4f} ± {result['std']:.4f}"
        )

iterations= 50 | lr=0.01 | CV=0.8013 ± 0.0238
iterations= 50 | lr=0.03 | CV=0.8159 ± 0.0167
iterations= 50 | lr=0.05 | CV=0.8260 ± 0.0118
iterations= 50 | lr=0.10 | CV=0.8294 ± 0.0210
iterations= 50 | lr=0.20 | CV=0.8271 ± 0.0223
iterations= 50 | lr=0.30 | CV=0.8283 ± 0.0167
iterations=100 | lr=0.01 | CV=0.8103 ± 0.0138
iterations=100 | lr=0.03 | CV=0.8238 ± 0.0165
iterations=100 | lr=0.05 | CV=0.8305 ± 0.0098
iterations=100 | lr=0.10 | CV=0.8361 ± 0.0219
iterations=100 | lr=0.20 | CV=0.8372 ± 0.0245
iterations=100 | lr=0.30 | CV=0.8339 ± 0.0239
iterations=200 | lr=0.01 | CV=0.8328 ± 0.0105
iterations=200 | lr=0.03 | CV=0.8350 ± 0.0163
iterations=200 | lr=0.05 | CV=0.8327 ± 0.0201
iterations=200 | lr=0.10 | CV=0.8339 ± 0.0249
iterations=200 | lr=0.20 | CV=0.8428 ± 0.0277
iterations=200 | lr=0.30 | CV=0.8395 ± 0.0192
iterations=500 | lr=0.01 | CV=0.8316 ± 0.0103
iterations=500 | lr=0.03 | CV=0.8361 ± 0.0264
iterations=500 | lr=0.05 | CV=0.8350 ± 0.0223
iterations=500 | lr=0.10 | CV=0.83

### Interaction between iterations and learning rate

Since `iterations` and `learning_rate` are closely related in gradient
boosting, both hyperparameters were evaluated jointly.

Low learning rates required more boosting iterations to achieve competitive
performance. For example, with `learning_rate=0.01`, increasing the number
of iterations from 50 to 200 improved mean CV accuracy from 0.8013 to 0.8328.

Higher learning rates reached stronger performance with fewer iterations.
The highest mean CV accuracy among the tested combinations was obtained with
`iterations=200` and `learning_rate=0.20` (0.8428 ± 0.0277).

However, several configurations produced similar results relative to the
cross-validation variability, so this result should not be interpreted as
evidence of a uniquely optimal configuration.

In [10]:
depths = [2, 3, 4, 5, 6, 7, 8, 10]

for depth in depths:
    result = evaluate_catboost(
        X=X,
        y=y,
        cat_features=CAT_FEATURES,
        model_params={
            "iterations": 200,
            "learning_rate": 0.2,
            "depth": depth,
        },
        seed=SEED,
    )

    print(
        f"depth={depth:2d} | "
        f"CV={result['mean']:.4f} ± {result['std']:.4f}"
    )

depth= 2 | CV=0.8316 ± 0.0190
depth= 3 | CV=0.8327 ± 0.0263
depth= 4 | CV=0.8361 ± 0.0225
depth= 5 | CV=0.8327 ± 0.0233
depth= 6 | CV=0.8428 ± 0.0277
depth= 7 | CV=0.8350 ± 0.0249
depth= 8 | CV=0.8215 ± 0.0244
depth=10 | CV=0.8327 ± 0.0251


### Tree depth

Tree depth was evaluated while keeping `iterations=200` and
`learning_rate=0.20` fixed.

The highest mean CV accuracy among the tested values was obtained with
`depth=6` (0.8428 ± 0.0277).

Increasing the depth beyond 6 did not improve cross-validation performance.
In particular, `depth=8` resulted in a noticeable decrease in mean CV
accuracy.

This suggests that increasing tree complexity beyond a moderate depth does
not provide additional generalization benefit for this dataset. However,
the differences between several configurations are relatively small
compared with the cross-validation variability.

In [11]:
l2_values = [1, 3, 5, 7, 10, 20, 50]

for l2 in l2_values:
    result = evaluate_catboost(
        X=X,
        y=y,
        cat_features=CAT_FEATURES,
        model_params={
            "iterations": 200,
            "learning_rate": 0.2,
            "depth": 6,
            "l2_leaf_reg": l2,
        },
        seed=SEED,
    )

    print(
        f"l2_leaf_reg={l2:2d} | "
        f"CV={result['mean']:.4f} ± {result['std']:.4f}"
    )

l2_leaf_reg= 1 | CV=0.8361 ± 0.0230
l2_leaf_reg= 3 | CV=0.8428 ± 0.0277
l2_leaf_reg= 5 | CV=0.8339 ± 0.0320
l2_leaf_reg= 7 | CV=0.8372 ± 0.0260
l2_leaf_reg=10 | CV=0.8428 ± 0.0374
l2_leaf_reg=20 | CV=0.8294 ± 0.0319
l2_leaf_reg=50 | CV=0.8406 ± 0.0210


# LightGBM approach

## LightGBM

LightGBM is a gradient boosting framework based on decision trees.

Unlike CatBoost, categorical features in this experiment are encoded during
preprocessing. Numerical features are passed through without scaling.

Goals:
- establish a LightGBM baseline;
- understand its main hyperparameters;
- evaluate it using stratified cross-validation;
- perform lightweight hyperparameter tuning;
- compare it with CatBoost and previous models.

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline

from src.preprocessing import build_preprocessor
from src.validation import evaluate_model

In [6]:
preprocessor = build_preprocessor('boosting')

model = get_model('lightgbm', seed=SEED)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model),
])

result = evaluate_model(
    estimator=pipeline,
    X=X,
    y=y,
    n_splits=5,
    scoring="accuracy",
    seed=SEED,
)

result

{'scores': array([0.86592179, 0.85393258, 0.80337079, 0.81460674, 0.84831461]),
 'mean': np.float64(0.8372293013621241),
 'std': np.float64(0.024013836290678183)}

In [9]:
def objective_lgbm(trial):
    """Objective function for LightGBM hyperparameter optimization."""
    params = {
        "n_estimators": 500,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "num_leaves": trial.suggest_int("num_leaves", 7, 63),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 60),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        'subsample_freq': 1,
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }

    model = get_model('lightgbm', seed=SEED, **params)

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])

    result = evaluate_model(
        estimator=pipeline,
        X=X,
        y=y,
        n_splits=5,
        scoring="accuracy",
        seed=SEED,
    )

    trial.set_user_attr("cv_std", result["std"])

    return result["mean"]

In [8]:
import optuna

c:\Users\coffe\miniconda3\envs\titanic-ml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
sampler = optuna.samplers.TPESampler(seed=SEED)

study_lgbm = optuna.create_study(direction="maximize", sampler=sampler)

study_lgbm.optimize(objective_lgbm, n_trials=30)

[I 2026-10-01 13:31:53,003] A new study created in memory with name: no-name-f6f3504f-3918-49e9-8e5a-be90975d5269
[I 2026-10-01 13:31:53,476] Trial 0 finished with value: 0.8260121775155357 and parameters: {'learning_rate': 0.030710573677773714, 'max_depth': 8, 'num_leaves': 48, 'min_child_samples': 40, 'subsample': 0.5780093202212182, 'colsample_bytree': 0.5779972601681014, 'reg_alpha': 0.0017073967431528124, 'reg_lambda': 2.9154431891537547}. Best is trial 0 with value: 0.8260121775155357.
[I 2026-10-01 13:31:53,912] Trial 1 finished with value: 0.8293704098926622 and parameters: {'learning_rate': 0.06054365855469246, 'max_depth': 7, 'num_leaves': 8, 'min_child_samples': 59, 'subsample': 0.9162213204002109, 'colsample_bytree': 0.6061695553391381, 'reg_alpha': 0.005337032762603957, 'reg_lambda': 0.00541524411940254}. Best is trial 1 with value: 0.8293704098926622.
[I 2026-10-01 13:31:54,503] Trial 2 finished with value: 0.8383528968677421 and parameters: {'learning_rate': 0.0248787344

In [11]:
print(study_lgbm.best_value)
print(study_lgbm.best_params)

0.84509446990145
{'learning_rate': 0.022174667089106155, 'max_depth': 7, 'num_leaves': 44, 'min_child_samples': 21, 'subsample': 0.9999223965727282, 'colsample_bytree': 0.7076938307243427, 'reg_alpha': 0.24522972168566148, 'reg_lambda': 0.0020199184365372576}


In [12]:
lgbm_trials = study_lgbm.trials_dataframe()

lgbm_trials.sort_values(
    "value",
    ascending=False,
).head(10)

,number,value,datetime_start,datetime_complete,duration,params_colsample_bytree,params_learning_rate,params_max_depth,params_min_child_samples,params_num_leaves,params_reg_alpha,params_reg_lambda,params_subsample,user_attrs_cv_std,system_attrs_tpe:relative_params:0,state
26,26,0.845094,2026-10-01 13:32:09.011148,2026-10-01 13:32:09.629446,0 days 00:00:00.618298,0.707694,0.022175,7,21,44,0.245230,0.002020,0.999922,0.024895,"{""colsample_bytree"": 0.7076938307243427, ""lear...",COMPLETE
15,15,0.843977,2026-10-01 13:32:00.721719,2026-10-01 13:32:01.398412,0 days 00:00:00.676693,0.590904,0.017744,7,18,36,0.130159,0.011499,0.991712,0.029256,"{""colsample_bytree"": 0.5909038387112041, ""lear...",COMPLETE
22,22,0.843971,2026-10-01 13:32:05.669228,2026-10-01 13:32:06.528136,0 days 00:00:00.858908,0.689512,0.010929,7,10,40,0.068897,0.002675,0.993501,0.016634,"{""colsample_bytree"": 0.6895116323028501, ""lear...",COMPLETE
25,25,0.842847,2026-10-01 13:32:08.331236,2026-10-01 13:32:09.009145,0 days 00:00:00.677909,0.686087,0.016112,7,19,45,0.182789,0.001416,0.961470,0.028100,"{""colsample_bytree"": 0.6860865893848698, ""lear...",COMPLETE
18,18,0.842847,2026-10-01 13:32:02.694339,2026-10-01 13:32:03.638531,0 days 00:00:00.944192,0.713300,0.011400,7,13,48,0.170115,0.021857,0.956670,0.020579,"{""colsample_bytree"": 0.7132998024912667, ""lear...",COMPLETE
10,10,0.842841,2026-10-01 13:31:57.752697,2026-10-01 13:31:58.367200,0 days 00:00:00.614503,0.694637,0.026633,6,17,25,0.495176,2.293803,0.838536,0.027442,"{""colsample_bytree"": 0.6946370554286224, ""lear...",COMPLETE
20,20,0.841724,2026-10-01 13:32:04.319526,2026-10-01 13:32:05.068784,0 days 00:00:00.749258,0.540712,0.019886,8,22,40,0.053033,0.013574,0.968425,0.027971,"{""colsample_bytree"": 0.5407122855030744, ""lear...",COMPLETE
17,17,0.841724,2026-10-01 13:32:02.081389,2026-10-01 13:32:02.692334,0 days 00:00:00.610945,0.686007,0.040050,6,23,32,0.219346,0.002957,0.921531,0.027053,"{""colsample_bytree"": 0.686006757559291, ""learn...",COMPLETE
27,27,0.841717,2026-10-01 13:32:09.630446,2026-10-01 13:32:10.428625,0 days 00:00:00.798179,0.564347,0.011069,7,11,38,0.306263,0.004963,0.950109,0.019485,"{""colsample_bytree"": 0.5643466547965059, ""lear...",COMPLETE
23,23,0.840600,2026-10-01 13:32:06.529640,2026-10-01 13:32:07.556958,0 days 00:00:01.027318,0.639717,0.010335,8,10,37,0.058857,0.001043,0.973051,0.021080,"{""colsample_bytree"": 0.6397166484739817, ""lear...",COMPLETE


In [13]:
best_lgbm_params = {
    **study_lgbm.best_params,
    "n_estimators": 500,
    "subsample_freq": 1,
}

# XGBoost approach

In [14]:
xgb_model = get_model(
    "xgboost",
    seed=SEED,
)

xgb_pipeline = Pipeline([
    ("preprocessor", build_preprocessor("boosting")),
    ("model", xgb_model),
])

xgb_baseline = evaluate_model(
    estimator=xgb_pipeline,
    X=X,
    y=y,
    n_splits=5,
    scoring="accuracy",
    seed=SEED,
)

xgb_baseline

{'scores': array([0.84916201, 0.80898876, 0.78651685, 0.79775281, 0.83146067]),
 'mean': np.float64(0.814776222459356),
 'std': np.float64(0.02272723733564801)}

In [18]:
def objective_xgb(trial):
    params = {
        'n_estimators': 500,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth': trial.suggest_int('max_depth', 2, 8),
        'min_child_weight': trial.suggest_int('min_child_weight', 1.0, 20.0, log=True),
        'gamma': trial.suggest_float('gamma', 1e-3, 5.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
    }
    
    model = get_model('xgboost', seed=SEED, **params)
    
    pipeline = Pipeline([
        ("preprocessor", build_preprocessor("boosting")),
        ("model", model),
    ])
    
    result = evaluate_model(
        estimator=pipeline,
        X=X,
        y=y,
        n_splits=5,
        scoring="accuracy",
        seed=SEED,
    )
    
    trial.set_user_attr("cv_std", result["std"])

    return result["mean"]

In [19]:
sampler = optuna.samplers.TPESampler(seed=SEED)

study_xgb = optuna.create_study(direction="maximize", sampler=sampler)

study_xgb.optimize(objective_xgb, n_trials=30)

[I 2026-10-01 14:02:49,053] A new study created in memory with name: no-name-2eb9efe8-2664-4a38-92c6-f99aefa41358
[I 2026-10-01 14:02:49,806] Trial 0 finished with value: 0.8237649865042999 and parameters: {'learning_rate': 0.030710573677773714, 'max_depth': 8, 'min_child_weight': 8, 'gamma': 0.16383993835282307, 'reg_alpha': 0.004207988669606638, 'reg_lambda': 0.004207053950287938, 'subsample': 0.6232334448672797, 'colsample_bytree': 0.9464704583099741}. Best is trial 0 with value: 0.8237649865042999.
[I 2026-10-01 14:02:50,376] Trial 1 finished with value: 0.8204381394764925 and parameters: {'learning_rate': 0.06054365855469246, 'max_depth': 6, 'min_child_weight': 1, 'gamma': 3.8696122574142664, 'reg_alpha': 2.1368329072358767, 'reg_lambda': 0.0070689749506246055, 'subsample': 0.6727299868828402, 'colsample_bytree': 0.6733618039413735}. Best is trial 0 with value: 0.8237649865042999.
[I 2026-10-01 14:02:51,476] Trial 2 finished with value: 0.8406126420187057 and parameters: {'learnin

In [20]:
print("Best CV:", study_xgb.best_value)
print("Best params:", study_xgb.best_params)

Best CV: 0.8439708743958321
Best params: {'learning_rate': 0.010001820723260776, 'max_depth': 8, 'min_child_weight': 2, 'gamma': 0.045456631012784354, 'reg_alpha': 0.3238480600985586, 'reg_lambda': 0.01581667373072451, 'subsample': 0.8764722160325104, 'colsample_bytree': 0.6189721852612399}


In [21]:
xgb_trials = study_xgb.trials_dataframe()

xgb_trials.sort_values(
    "value",
    ascending=False,
).head(10)

,number,value,datetime_start,datetime_complete,duration,params_colsample_bytree,params_gamma,params_learning_rate,params_max_depth,params_min_child_weight,params_reg_alpha,params_reg_lambda,params_subsample,user_attrs_cv_std,system_attrs_tpe:relative_params:0,state
23,23,0.843971,2026-10-01 14:03:08.193728,2026-10-01 14:03:09.489328,0 days 00:00:01.295600,0.618972,0.045457,0.010002,8,2,0.323848,0.015817,0.876472,0.023813,"{""colsample_bytree"": 0.6189721852612399, ""gamm...",COMPLETE
24,24,0.843965,2026-10-01 14:03:09.490334,2026-10-01 14:03:10.677121,0 days 00:00:01.186787,0.602760,0.025161,0.020919,7,1,0.284930,0.008744,0.811535,0.024105,"{""colsample_bytree"": 0.6027602184330864, ""gamm...",COMPLETE
25,25,0.843965,2026-10-01 14:03:10.678122,2026-10-01 14:03:11.671155,0 days 00:00:00.993033,0.607803,0.023044,0.014824,6,1,1.221791,0.016617,0.785461,0.023575,"{""colsample_bytree"": 0.6078028289080123, ""gamm...",COMPLETE
20,20,0.842847,2026-10-01 14:03:04.982719,2026-10-01 14:03:06.074184,0 days 00:00:01.091465,0.621212,0.028713,0.016914,7,1,1.375564,0.162763,0.889060,0.026953,"{""colsample_bytree"": 0.6212119051204478, ""gamm...",COMPLETE
19,19,0.841711,2026-10-01 14:03:03.913433,2026-10-01 14:03:04.981718,0 days 00:00:01.068285,0.652351,0.029531,0.036768,7,1,0.630340,0.049060,0.835057,0.028902,"{""colsample_bytree"": 0.6523510266579359, ""gamm...",COMPLETE
2,2,0.840613,2026-10-01 14:02:50.377821,2026-10-01 14:02:51.476140,0 days 00:00:01.098319,0.746545,0.011947,0.024879,5,2,0.280164,0.003614,0.716858,0.029930,NaN,COMPLETE
21,21,0.840613,2026-10-01 14:03:06.076185,2026-10-01 14:03:07.027991,0 days 00:00:00.951806,0.667106,0.038765,0.044492,7,1,1.580495,0.409197,0.849263,0.025857,"{""colsample_bytree"": 0.6671063521386916, ""gamm...",COMPLETE
11,11,0.840606,2026-10-01 14:02:57.282107,2026-10-01 14:02:58.126763,0 days 00:00:00.844656,0.664675,0.171254,0.042226,7,2,0.571107,0.004289,0.913224,0.026845,"{""colsample_bytree"": 0.6646747841131975, ""gamm...",COMPLETE
22,22,0.839483,2026-10-01 14:03:07.029497,2026-10-01 14:03:08.191729,0 days 00:00:01.162232,0.647718,0.002484,0.011019,8,1,1.705463,0.132396,0.903352,0.021935,"{""colsample_bytree"": 0.6477180281081257, ""gamm...",COMPLETE
29,29,0.838347,2026-10-01 14:03:15.532233,2026-10-01 14:03:16.910413,0 days 00:00:01.378180,0.601762,0.032434,0.014608,6,1,1.727892,0.012024,0.803397,0.028680,"{""colsample_bytree"": 0.6017618112540882, ""gamm...",COMPLETE


In [22]:
best_xgb_params = {
    **study_xgb.best_params,
    "n_estimators": 500,
}